# Zadanie 5: programowanie genetyczne i regresja symboliczna

Termin realizacji: 18 maja 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^": [https://astroautomata.com/PySR/v1.5.9/operators#custom](https://astroautomata.com/PySR/v1.5.9/operators#custom) ).
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


## Importy

In [ ]:
import numpy as np
import sympy
from pysr import PySRRegressor


In [ ]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
    niterations=40,
)


## Funkcja pomocnicza do wyświetlania wyników

In [ ]:
def show_results(model):
    display(model.equations_.nlargest(3, 'score')[['complexity', 'loss', 'score', 'equation']])
    print('best:', model.sympy())


## 3.0 — Dane bez szumu, zakres [-5, 5]

Funkcja: $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - 3$, dziedzina $\mathbb{R}^6$, 200 próbek.

In [ ]:
np.random.seed(0)
X = np.random.uniform(-5, 5, (200, 6))
y = 2.2 * np.sin(X[:, 0] + 2 * X[:, 1]) - X[:, 5]**2 - 3


### Konfiguracja 1 — `binary=["+","*"]`, `unary=["cos","exp","sin"]`, `maxsize=20`

In [ ]:
model1 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1.fit(X, y)


In [ ]:
show_results(model1)


### Konfiguracja 2 — `binary=["+","*","-","^"]`, `unary=["cos","exp","sin","log"]`, `maxsize=30`

Ograniczenie dla `^`: prawy argument ma maksymalną złożoność 1.

In [ ]:
model2 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model2.fit(X, y)


In [ ]:
show_results(model2)


### Konfiguracja 3 — `binary=["+","*","-","^"]`, `unary=["exp","sin"]`, `maxsize=15`

In [ ]:
model3 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model3.fit(X, y)


In [ ]:
show_results(model3)


## 3.0 — Dane z szumem $\mathcal{N}(0,\,0.5^2)$, zakres [-5, 5]

In [ ]:
np.random.seed(0)
X_n05 = np.random.uniform(-5, 5, (200, 6))
y_n05 = 2.2 * np.sin(X_n05[:, 0] + 2 * X_n05[:, 1]) - X_n05[:, 5]**2 - 3 + np.random.normal(0, 0.5, 200)


### Konfiguracja 1 (szum 0.5)

In [ ]:
model1_n05 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_n05.fit(X_n05, y_n05)


In [ ]:
show_results(model1_n05)


### Konfiguracja 2 (szum 0.5)

In [ ]:
model2_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model2_n05.fit(X_n05, y_n05)


In [ ]:
show_results(model2_n05)


### Konfiguracja 3 (szum 0.5)

In [ ]:
model3_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model3_n05.fit(X_n05, y_n05)


In [ ]:
show_results(model3_n05)


## 4.0 — Zakres [-15, 15], szum $\sigma=2$

In [ ]:
np.random.seed(0)
X_w2 = np.random.uniform(-15, 15, (200, 6))
y_w2 = 2.2 * np.sin(X_w2[:, 0] + 2 * X_w2[:, 1]) - X_w2[:, 5]**2 - 3 + np.random.normal(0, 2, 200)


### Konfiguracja 1 (zakres [-15,15], σ=2)

In [ ]:
model1_w2 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_w2.fit(X_w2, y_w2)


In [ ]:
show_results(model1_w2)


### Konfiguracja 2 (zakres [-15,15], σ=2)

In [ ]:
model2_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model2_w2.fit(X_w2, y_w2)


In [ ]:
show_results(model2_w2)


### Konfiguracja 3 (zakres [-15,15], σ=2)

In [ ]:
model3_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model3_w2.fit(X_w2, y_w2)


In [ ]:
show_results(model3_w2)


## 4.0 — Zakres [-15, 15], szum $\sigma=5$

In [ ]:
np.random.seed(0)
X_w5 = np.random.uniform(-15, 15, (200, 6))
y_w5 = 2.2 * np.sin(X_w5[:, 0] + 2 * X_w5[:, 1]) - X_w5[:, 5]**2 - 3 + np.random.normal(0, 5, 200)


### Konfiguracja 1 (zakres [-15,15], σ=5)

In [ ]:
model1_w5 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_w5.fit(X_w5, y_w5)


In [ ]:
show_results(model1_w5)


### Konfiguracja 2 (zakres [-15,15], σ=5)

In [ ]:
model2_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model2_w5.fit(X_w5, y_w5)


In [ ]:
show_results(model2_w5)


### Konfiguracja 3 (zakres [-15,15], σ=5)

In [ ]:
model3_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    **default_pysr_params,
)
model3_w5.fit(X_w5, y_w5)


In [ ]:
show_results(model3_w5)


## 5.0 — Operator liczb pierwszych $p(i)$

Funkcja: $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$

### Instalacja Primes.jl i definicja operatora `p`

In [ ]:
from pysr import jl

jl.seval("""
import Pkg
Pkg.add(\"Primes\")
""")


In [ ]:
jl.seval("using Primes: prime")


In [ ]:
jl.seval("""
function p(i::T) where T
    if 0.5 < i < 1000
        return T(prime(round(Int, i)))
    else
        return T(NaN)
    end
end
""")


Definicja `sympy_p` oraz pomocnicza funkcja Pythonowa do generowania próbek.

In [ ]:
class sympy_p(sympy.Function):
    pass

def p_python(x0_val):
    i = int(np.floor(x0_val))
    if 1 <= i <= 999:
        return float(sympy.prime(i))
    return np.nan


In [ ]:
def make_data_prime(low, high, n=200, noise_std=0.0, seed=0):
    rng = np.random.default_rng(seed)
    # generujemy więcej próbek, bo część x0 daje NaN
    X_raw = rng.uniform(low, high, (n * 6, 6))
    p_vals = np.array([p_python(x) for x in X_raw[:, 0]])
    y_raw = 2.2 * np.sin(X_raw[:, 0] + 2 * X_raw[:, 1]) - X_raw[:, 5]**2 - p_vals
    if noise_std > 0:
        y_raw += rng.normal(0, noise_std, len(y_raw))
    mask = ~np.isnan(y_raw)
    return X_raw[mask][:n], y_raw[mask][:n]


### 5.0 — Bez szumu, zakres [-5, 5]

In [ ]:
X_p, y_p = make_data_prime(-5, 5)
print(f'Próbek: {len(y_p)}')


### Konfiguracja 1 z operatorem `p` (bez szumu)

In [ ]:
model1_p = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p.fit(X_p, y_p)


In [ ]:
show_results(model1_p)


### Konfiguracja 2 z operatorem `p` (bez szumu)

In [ ]:
model2_p = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model2_p.fit(X_p, y_p)


In [ ]:
show_results(model2_p)


### Konfiguracja 3 z operatorem `p` (bez szumu)

In [ ]:
model3_p = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model3_p.fit(X_p, y_p)


In [ ]:
show_results(model3_p)


### 5.0 — Szum $\sigma=0.5$, zakres [-5, 5]

In [ ]:
X_p_n05, y_p_n05 = make_data_prime(-5, 5, noise_std=0.5)


### Konfiguracja 1 z operatorem `p` (szum 0.5)

In [ ]:
model1_p_n05 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_n05.fit(X_p_n05, y_p_n05)


In [ ]:
show_results(model1_p_n05)


### Konfiguracja 2 z operatorem `p` (szum 0.5)

In [ ]:
model2_p_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model2_p_n05.fit(X_p_n05, y_p_n05)


In [ ]:
show_results(model2_p_n05)


### Konfiguracja 3 z operatorem `p` (szum 0.5)

In [ ]:
model3_p_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model3_p_n05.fit(X_p_n05, y_p_n05)


In [ ]:
show_results(model3_p_n05)


### 5.0 — Zakres [-15, 15], szum $\sigma=2$

In [ ]:
X_p_w2, y_p_w2 = make_data_prime(-15, 15, noise_std=2)


### Konfiguracja 1 z operatorem `p` (zakres [-15,15], σ=2)

In [ ]:
model1_p_w2 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_w2.fit(X_p_w2, y_p_w2)


In [ ]:
show_results(model1_p_w2)


### Konfiguracja 2 z operatorem `p` (zakres [-15,15], σ=2)

In [ ]:
model2_p_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model2_p_w2.fit(X_p_w2, y_p_w2)


In [ ]:
show_results(model2_p_w2)


### Konfiguracja 3 z operatorem `p` (zakres [-15,15], σ=2)

In [ ]:
model3_p_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model3_p_w2.fit(X_p_w2, y_p_w2)


In [ ]:
show_results(model3_p_w2)


### 5.0 — Zakres [-15, 15], szum $\sigma=5$

In [ ]:
X_p_w5, y_p_w5 = make_data_prime(-15, 15, noise_std=5)


### Konfiguracja 1 z operatorem `p` (zakres [-15,15], σ=5)

In [ ]:
model1_p_w5 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_w5.fit(X_p_w5, y_p_w5)


In [ ]:
show_results(model1_p_w5)


### Konfiguracja 2 z operatorem `p` (zakres [-15,15], σ=5)

In [ ]:
model2_p_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model2_p_w5.fit(X_p_w5, y_p_w5)


In [ ]:
show_results(model2_p_w5)


### Konfiguracja 3 z operatorem `p` (zakres [-15,15], σ=5)

In [ ]:
model3_p_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"^": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model3_p_w5.fit(X_p_w5, y_p_w5)


In [ ]:
show_results(model3_p_w5)
